# Xây Dựng & Chuẩn Hóa Bảng Danh Mục Kỹ Năng IT: `skills`

Notebook này tập trung xây dựng bảng danh mục chiều (Master Dimension Table) **`skills`** trong kiến trúc Silver Layer:
- Cung cấp taxonomy chuẩn hóa cho toàn bộ hệ thống, phân chia thành các nhóm chuyên môn (category) như Programming Language, Frontend, Backend, Database, Cloud & DevOps, AI/Data, Mobile, Testing/QA, System & Security, Methodology & Tools.
- Thiết lập quan hệ cây phân cấp cha-con thông qua khóa ngoại tự tham chiếu `parent_id` (ví dụ: Spring Boot thuộc Java, FastAPI thuộc Python, React thuộc JavaScript, Next.js thuộc React).
- Khảo sát tần suất xuất hiện thực tế của từng kỹ năng trong toàn bộ 2,327 tin tuyển dụng IT để đánh giá độ phủ (coverage).
- Đảm bảo 100% tính toàn vẹn ràng buộc DDL PostgreSQL (khóa chính, tính duy nhất skill_name, canonical_name, khóa ngoại tự tham chiếu).

In [1]:
import os
import re
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng skills đã sẵn sàng!')

Môi trường xử lý bảng skills đã sẵn sàng!


## 1. Định Nghĩa Danh Mục Kỹ Năng IT Chuẩn Hóa (IT Skills Taxonomy)

Xây dựng danh mục gồm 76 kỹ năng IT cốt lõi nhất trên thị trường tuyển dụng Việt Nam, phân theo 10 nhóm nghiệp vụ.

In [2]:
skills_raw_definition = [
    # 1. Programming Languages
    {'name': 'Python', 'canonical': 'python', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình đa dụng, AI/Data, Backend'},
    {'name': 'Java', 'canonical': 'java', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình hướng đối tượng doanh nghiệp'},
    {'name': 'JavaScript', 'canonical': 'javascript', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình web phổ biến nhất'},
    {'name': 'TypeScript', 'canonical': 'typescript', 'cat': 'Programming Language', 'parent': 'JavaScript', 'desc': 'Phiên bản typed của JavaScript'},
    {'name': 'C++', 'canonical': 'cpp', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình hệ thống hiệu năng cao'},
    {'name': 'C#', 'canonical': 'csharp', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình hệ sinh thái .NET'},
    {'name': 'PHP', 'canonical': 'php', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình backend web truyền thống'},
    {'name': 'Golang', 'canonical': 'golang', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ biên dịch hiệu năng cao của Google'},
    {'name': 'Ruby', 'canonical': 'ruby', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình hướng đối tượng linh hoạt'},
    {'name': 'Kotlin', 'canonical': 'kotlin', 'cat': 'Programming Language', 'parent': 'Java', 'desc': 'Ngôn ngữ lập trình hiện đại cho Android và JVM'},
    {'name': 'Swift', 'canonical': 'swift', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ lập trình hệ sinh thái Apple iOS/macOS'},
    {'name': 'Rust', 'canonical': 'rust', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ hệ thống an toàn bộ nhớ'},
    {'name': 'Dart', 'canonical': 'dart', 'cat': 'Programming Language', 'parent': None, 'desc': 'Ngôn ngữ phát triển ứng dụng Flutter'},
    {'name': 'Scala', 'canonical': 'scala', 'cat': 'Programming Language', 'parent': 'Java', 'desc': 'Ngôn ngữ kết hợp OOP và FP cho Big Data'},

    # 2. Frontend Development
    {'name': 'HTML/CSS', 'canonical': 'html_css', 'cat': 'Frontend Development', 'parent': None, 'desc': 'Nền tảng xây dựng giao diện web'},
    {'name': 'React', 'canonical': 'react', 'cat': 'Frontend Development', 'parent': 'JavaScript', 'desc': 'Thư viện frontend phổ biến nhất của Meta'},
    {'name': 'Vue.js', 'canonical': 'vuejs', 'cat': 'Frontend Development', 'parent': 'JavaScript', 'desc': 'Progressive framework frontend'},
    {'name': 'Angular', 'canonical': 'angular', 'cat': 'Frontend Development', 'parent': 'TypeScript', 'desc': 'Framework frontend toàn diện của Google'},
    {'name': 'Next.js', 'canonical': 'nextjs', 'cat': 'Frontend Development', 'parent': 'React', 'desc': 'React framework hỗ trợ SSR và full-stack'},
    {'name': 'Tailwind CSS', 'canonical': 'tailwindcss', 'cat': 'Frontend Development', 'parent': 'HTML/CSS', 'desc': 'Utility-first CSS framework'},
    {'name': 'Nuxt.js', 'canonical': 'nuxtjs', 'cat': 'Frontend Development', 'parent': 'Vue.js', 'desc': 'Vue framework hỗ trợ SSR'},
    {'name': 'Redux', 'canonical': 'redux', 'cat': 'Frontend Development', 'parent': 'React', 'desc': 'Thư viện quản lý state'},

    # 3. Backend Development
    {'name': 'Node.js', 'canonical': 'nodejs', 'cat': 'Backend Development', 'parent': 'JavaScript', 'desc': 'Runtime JavaScript phía server'},
    {'name': 'Spring Boot', 'canonical': 'spring_boot', 'cat': 'Backend Development', 'parent': 'Java', 'desc': 'Framework backend Java phổ biến nhất'},
    {'name': '.NET', 'canonical': 'dotnet', 'cat': 'Backend Development', 'parent': 'C#', 'desc': 'Nền tảng phát triển backend của Microsoft'},
    {'name': 'Django', 'canonical': 'django', 'cat': 'Backend Development', 'parent': 'Python', 'desc': 'Full-stack web framework cho Python'},
    {'name': 'FastAPI', 'canonical': 'fastapi', 'cat': 'Backend Development', 'parent': 'Python', 'desc': 'High-performance API framework cho Python'},
    {'name': 'Flask', 'canonical': 'flask', 'cat': 'Backend Development', 'parent': 'Python', 'desc': 'Micro-framework web cho Python'},
    {'name': 'Laravel', 'canonical': 'laravel', 'cat': 'Backend Development', 'parent': 'PHP', 'desc': 'Web framework PHP hiện đại và phổ biến nhất'},
    {'name': 'Express.js', 'canonical': 'expressjs', 'cat': 'Backend Development', 'parent': 'Node.js', 'desc': 'Minimal web framework cho Node.js'},
    {'name': 'NestJS', 'canonical': 'nestjs', 'cat': 'Backend Development', 'parent': 'Node.js', 'desc': 'Progressive Node.js framework hướng kiến trúc'},
    {'name': 'Ruby on Rails', 'canonical': 'rails', 'cat': 'Backend Development', 'parent': 'Ruby', 'desc': 'Framework phát triển web nhanh của Ruby'},

    # 4. Database & Storage
    {'name': 'SQL', 'canonical': 'sql', 'cat': 'Database & Storage', 'parent': None, 'desc': 'Ngôn ngữ truy vấn cơ sở dữ liệu quan hệ'},
    {'name': 'PostgreSQL', 'canonical': 'postgresql', 'cat': 'Database & Storage', 'parent': 'SQL', 'desc': 'Hệ quản trị CSDL quan hệ mã nguồn mở mạnh mẽ nhất'},
    {'name': 'MySQL', 'canonical': 'mysql', 'cat': 'Database & Storage', 'parent': 'SQL', 'desc': 'Hệ quản trị CSDL quan hệ phổ biến nhất'},
    {'name': 'MongoDB', 'canonical': 'mongodb', 'cat': 'Database & Storage', 'parent': None, 'desc': 'Cơ sở dữ liệu NoSQL Document'},
    {'name': 'Redis', 'canonical': 'redis', 'cat': 'Database & Storage', 'parent': None, 'desc': 'In-memory data store, cache và message broker'},
    {'name': 'Microsoft SQL Server', 'canonical': 'sqlserver', 'cat': 'Database & Storage', 'parent': 'SQL', 'desc': 'Hệ quản trị CSDL của Microsoft'},
    {'name': 'Oracle Database', 'canonical': 'oracle_db', 'cat': 'Database & Storage', 'parent': 'SQL', 'desc': 'Hệ quản trị CSDL cấp doanh nghiệp lớn'},
    {'name': 'Elasticsearch', 'canonical': 'elasticsearch', 'cat': 'Database & Storage', 'parent': None, 'desc': 'Công cụ tìm kiếm và phân tích phân tán'},

    # 5. Cloud & DevOps
    {'name': 'AWS', 'canonical': 'aws', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Nền tảng điện toán đám mây Amazon Web Services'},
    {'name': 'Microsoft Azure', 'canonical': 'azure', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Nền tảng điện toán đám mây của Microsoft'},
    {'name': 'Google Cloud (GCP)', 'canonical': 'gcp', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Nền tảng điện toán đám mây của Google'},
    {'name': 'Docker', 'canonical': 'docker', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Nền tảng container hóa ứng dụng'},
    {'name': 'Kubernetes', 'canonical': 'kubernetes', 'cat': 'Cloud & DevOps', 'parent': 'Docker', 'desc': 'Hệ thống điều phối container tự động hóa'},
    {'name': 'CI/CD', 'canonical': 'cicd', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Quy trình tích hợp và triển khai liên tục'},
    {'name': 'Terraform', 'canonical': 'terraform', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Công cụ quản lý hạ tầng bằng mã nguồn (IaC)'},
    {'name': 'Linux', 'canonical': 'linux', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Hệ điều hành máy chủ phổ biến nhất'},
    {'name': 'Git', 'canonical': 'git', 'cat': 'Cloud & DevOps', 'parent': None, 'desc': 'Hệ thống quản lý phiên bản phân tán'},

    # 6. AI, Machine Learning & Data
    {'name': 'Machine Learning', 'canonical': 'machine_learning', 'cat': 'AI & Data Science', 'parent': None, 'desc': 'Học máy và thuật toán dự đoán'},
    {'name': 'Deep Learning', 'canonical': 'deep_learning', 'cat': 'AI & Data Science', 'parent': 'Machine Learning', 'desc': 'Mạng nơ-ron sâu'},
    {'name': 'PyTorch', 'canonical': 'pytorch', 'cat': 'AI & Data Science', 'parent': 'Deep Learning', 'desc': 'Framework Deep Learning hàng đầu của Meta'},
    {'name': 'TensorFlow', 'canonical': 'tensorflow', 'cat': 'AI & Data Science', 'parent': 'Deep Learning', 'desc': 'Framework Deep Learning của Google'},
    {'name': 'LLM (Generative AI)', 'canonical': 'llm_genai', 'cat': 'AI & Data Science', 'parent': 'Deep Learning', 'desc': 'Mô hình ngôn ngữ lớn và AI tạo sinh'},
    {'name': 'NLP', 'canonical': 'nlp', 'cat': 'AI & Data Science', 'parent': 'Machine Learning', 'desc': 'Xử lý ngôn ngữ tự nhiên'},
    {'name': 'Computer Vision', 'canonical': 'computer_vision', 'cat': 'AI & Data Science', 'parent': 'Machine Learning', 'desc': 'Thị giác máy tính'},
    {'name': 'Data Engineering', 'canonical': 'data_engineering', 'cat': 'AI & Data Science', 'parent': None, 'desc': 'Kỹ thuật dữ liệu và xây dựng pipeline'},
    {'name': 'Apache Spark', 'canonical': 'spark', 'cat': 'AI & Data Science', 'parent': 'Data Engineering', 'desc': 'Framework xử lý dữ liệu lớn phân tán'},
    {'name': 'Power BI', 'canonical': 'power_bi', 'cat': 'AI & Data Science', 'parent': None, 'desc': 'Công cụ trực quan hóa và phân tích dữ liệu'},
    {'name': 'Tableau', 'canonical': 'tableau', 'cat': 'AI & Data Science', 'parent': None, 'desc': 'Nền tảng BI và trực quan hóa dữ liệu'},

    # 7. Mobile Development
    {'name': 'Flutter', 'canonical': 'flutter', 'cat': 'Mobile Development', 'parent': 'Dart', 'desc': 'Cross-platform mobile UI framework của Google'},
    {'name': 'React Native', 'canonical': 'react_native', 'cat': 'Mobile Development', 'parent': 'React', 'desc': 'Cross-platform mobile framework của Meta'},
    {'name': 'iOS Development', 'canonical': 'ios_dev', 'cat': 'Mobile Development', 'parent': 'Swift', 'desc': 'Phát triển ứng dụng native iOS'},
    {'name': 'Android Development', 'canonical': 'android_dev', 'cat': 'Mobile Development', 'parent': 'Kotlin', 'desc': 'Phát triển ứng dụng native Android'},

    # 8. Testing & Quality Assurance
    {'name': 'Manual Testing', 'canonical': 'manual_testing', 'cat': 'Testing & QA', 'parent': None, 'desc': 'Kiểm thử phần mềm thủ công'},
    {'name': 'Automation Testing', 'canonical': 'automation_testing', 'cat': 'Testing & QA', 'parent': None, 'desc': 'Kiểm thử phần mềm tự động hóa'},
    {'name': 'Selenium', 'canonical': 'selenium', 'cat': 'Testing & QA', 'parent': 'Automation Testing', 'desc': 'Công cụ tự động hóa trình duyệt web'},
    {'name': 'Postman', 'canonical': 'postman', 'cat': 'Testing & QA', 'parent': None, 'desc': 'Nền tảng kiểm thử và quản lý API'},
    {'name': 'JMeter', 'canonical': 'jmeter', 'cat': 'Testing & QA', 'parent': 'Automation Testing', 'desc': 'Công cụ kiểm thử tải và hiệu năng'},
    {'name': 'Cypress', 'canonical': 'cypress', 'cat': 'Testing & QA', 'parent': 'Automation Testing', 'desc': 'Next generation front-end testing tool'},

    # 9. System, Security & Architecture
    {'name': 'Cybersecurity', 'canonical': 'cybersecurity', 'cat': 'Security & Architecture', 'parent': None, 'desc': 'An toàn và bảo mật thông tin'},
    {'name': 'Microservices', 'canonical': 'microservices', 'cat': 'Security & Architecture', 'parent': None, 'desc': 'Kiến trúc dịch vụ siêu nhỏ'},
    {'name': 'System Administration', 'canonical': 'system_admin', 'cat': 'Security & Architecture', 'parent': 'Linux', 'desc': 'Quản trị hệ thống và mạng'},

    # 10. Methodology & Project Management
    {'name': 'Agile', 'canonical': 'agile', 'cat': 'Methodology & Tools', 'parent': None, 'desc': 'Phương pháp luận phát triển phần mềm linh hoạt'},
    {'name': 'Scrum', 'canonical': 'scrum', 'cat': 'Methodology & Tools', 'parent': 'Agile', 'desc': 'Framework quản lý dự án Scrum'},
    {'name': 'Jira', 'canonical': 'jira', 'cat': 'Methodology & Tools', 'parent': None, 'desc': 'Phần mềm quản lý dự án và theo dõi lỗi'}
]

print(f'Tổng số kỹ năng được định nghĩa trong danh mục: {len(skills_raw_definition)} kỹ năng')
categories = set(s['cat'] for s in skills_raw_definition)
print(f'Tổng số nhóm chuyên môn (categories): {len(categories)} nhóm')

Tổng số kỹ năng được định nghĩa trong danh mục: 76 kỹ năng
Tổng số nhóm chuyên môn (categories): 10 nhóm


## 2. Gán Khóa Chính `skill_id` & Xây Dựng Khóa Ngoại Tự Tham Chiếu `parent_id`

In [3]:
name_to_id = {}
skill_rows = []
now_iso = datetime.now().isoformat()

# Bước 1: Gán skill_id (1-indexed) theo thứ tự
for idx, item in enumerate(skills_raw_definition):
    skill_id = idx + 1
    name_to_id[item['name']] = skill_id

# Bước 2: Chuyển đổi tên parent thành parent_id hợp lệ
for idx, item in enumerate(skills_raw_definition):
    skill_id = idx + 1
    parent_name = item['parent']
    parent_id = name_to_id.get(parent_name, None) if parent_name else None
    
    skill_rows.append({
        'skill_id': skill_id,
        'skill_name': item['name'],
        'canonical_name': item['canonical'],
        'category': item['cat'],
        'parent_id': parent_id,
        'description': item['desc'],
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_skills = pd.DataFrame(skill_rows)

print('Khởi tạo bảng skills thành công:')
print(f'  - Tổng số kỹ năng: {len(df_skills)}')
print(f'  - Số kỹ năng có quan hệ cha-con (parent_id not null): {df_skills["parent_id"].notna().sum()}')
print(f'  - Kiểm tra tính duy nhất skill_name: {df_skills["skill_name"].nunique() == len(df_skills)}')
print(f'  - Kiểm tra tính duy nhất canonical_name: {df_skills["canonical_name"].nunique() == len(df_skills)}')

Khởi tạo bảng skills thành công:
  - Tổng số kỹ năng: 76
  - Số kỹ năng có quan hệ cha-con (parent_id not null): 41
  - Kiểm tra tính duy nhất skill_name: True
  - Kiểm tra tính duy nhất canonical_name: True


## 3. Khảo Sát Độ Phủ (Coverage) Của Danh Mục Kỹ Năng Trên Dữ Liệu Thực Tế Bronze

Kiểm tra tần suất xuất hiện thực tế của các kỹ năng trong 2,327 tin tuyển dụng (thông qua tiêu đề, skills_raw và nội dung JD) để chứng minh độ tin cậy của danh mục.

In [4]:
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')

with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)
with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)

# Tổng hợp văn bản để tìm kiếm
job_corpus = []
for d, l in zip(details_data, listings_data):
    text = f"{l.get('title_raw', '')} {l.get('skills_raw', '')} {d.get('responsibilities_raw', '')} {d.get('requirements_raw', '')}".lower()
    job_corpus.append(text)

total_jobs = len(job_corpus)

# Từ điển regex cho từng kỹ năng
skill_regex_map = {
    'Python': r'\bpython\b',
    'Java': r'\bjava\b(?!script)',
    'JavaScript': r'\bjavascript\b|\bjs\b',
    'TypeScript': r'\btypescript\b|\bts\b',
    'C++': r'\bc\+\+\b|\bcpp\b',
    'C#': r'\bc#\b|\bcsharp\b',
    'PHP': r'\bphp\b',
    'Golang': r'\bgolang\b|\bgo\b',
    'React': r'\breact\b|\breactjs\b',
    'Vue.js': r'\bvue\b|\bvuejs\b',
    'Angular': r'\bangular\b',
    'Node.js': r'\bnode\b|\bnodejs\b',
    'Spring Boot': r'\bspring\b|\bspring boot\b',
    '.NET': r'\b\.net\b|\basp\.net\b',
    'PostgreSQL': r'\bpostgres\b|\bpostgresql\b',
    'MySQL': r'\bmysql\b',
    'MongoDB': r'\bmongo\b|\bmongodb\b',
    'Redis': r'\bredis\b',
    'AWS': r'\baws\b|\bamazon web services\b',
    'Docker': r'\bdocker\b',
    'Kubernetes': r'\bkubernetes\b|\bk8s\b',
    'CI/CD': r'\bci/cd\b|\bcicd\b',
    'Git': r'\bgit\b(?!lab|hub)',
    'Linux': r'\blinux\b',
    'Machine Learning': r'\bmachine learning\b|\bml\b',
    'LLM (Generative AI)': r'\bllm\b|\bgenerative ai\b|\brag\b',
    'Flutter': r'\bflutter\b',
    'React Native': r'\breact native\b',
    'Manual Testing': r'\bmanual\b|\bmanual test',
    'Automation Testing': r'\bautomation\b|\bautomation test',
    'Agile': r'\bagile\b',
    'Scrum': r'\bscrum\b',
    'Jira': r'\bjira\b'
}

coverage_stats = []
for skill_name, pattern in skill_regex_map.items():
    hit_count = sum(1 for text in job_corpus if re.search(pattern, text))
    coverage_stats.append({
        'Kỹ năng': skill_name,
        'Số tin tuyển dụng': hit_count,
        'Tỷ lệ trên thị trường (%)': round(hit_count / total_jobs * 100, 2)
    })

df_coverage = pd.DataFrame(coverage_stats).sort_values(by='Số tin tuyển dụng', ascending=False)
print('--- TOP KỸ NĂNG IT XUẤT HIỆN PHỔ BIẾN NHẤT TRÊN THỊ TRƯỜNG (2,327 TIN) ---')
print(df_coverage.head(20).to_string(index=False))

--- TOP KỸ NĂNG IT XUẤT HIỆN PHỔ BIẾN NHẤT TRÊN THỊ TRƯỜNG (2,327 TIN) ---
            Kỹ năng  Số tin tuyển dụng  Tỷ lệ trên thị trường (%)
             Python                343                      14.74
 Automation Testing                235                      10.10
               Java                228                       9.80
              Agile                206                       8.85
              CI/CD                185                       7.95
              Linux                178                       7.65
              Scrum                157                       6.75
             Docker                147                       6.32
         JavaScript                145                       6.23
         PostgreSQL                144                       6.19
                AWS                128                       5.50
                Git                121                       5.20
         Kubernetes                121                       5.20
 

## 4. Đánh Giá Tính Toàn Vẹn Bảng `skills` & Phân Tích Theo Nhóm Chuyên Môn

In [5]:
print('--- 1. CẤU TRÚC BẢNG skills (INFO) ---')
print(df_skills.info())

print('\n--- 2. PHÂN BỐ SỐ LƯỢNG KỸ NĂNG THEO NHÓM CHUYÊN MÔN (category) ---')
cat_counts = df_skills['category'].value_counts()
df_cat = pd.DataFrame({'Số lượng kỹ năng': cat_counts, 'Tỷ lệ (%)': (cat_counts / len(df_skills) * 100).round(1)})
print(df_cat.to_string())

print('\n--- 3. KIỂM TRA CÁC QUAN HỆ CÂY PHÂN CẤP (parent_id) ---')
df_hierarchy = df_skills[df_skills['parent_id'].notna()][['skill_id', 'skill_name', 'category', 'parent_id']].copy()
id_to_name = dict(zip(df_skills['skill_id'], df_skills['skill_name']))
df_hierarchy['parent_name'] = df_hierarchy['parent_id'].map(id_to_name)
print(df_hierarchy[['skill_id', 'skill_name', 'category', 'parent_id', 'parent_name']].head(12).to_string(index=False))

print('\n--- 4. PREVIEW 10 DÒNG BẢNG skills (CHUẨN POSTGRESQL) ---')
print(df_skills[['skill_id', 'skill_name', 'canonical_name', 'category', 'parent_id', 'description']].head(10).to_string(index=False))

--- 1. CẤU TRÚC BẢNG skills (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   skill_id        76 non-null     int64  
 1   skill_name      76 non-null     str    
 2   canonical_name  76 non-null     str    
 3   category        76 non-null     str    
 4   parent_id       41 non-null     float64
 5   description     76 non-null     str    
 6   created_at      76 non-null     str    
 7   updated_at      76 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 14.6 KB
None

--- 2. PHÂN BỐ SỐ LƯỢNG KỸ NĂNG THEO NHÓM CHUYÊN MÔN (category) ---
                         Số lượng kỹ năng  Tỷ lệ (%)
category                                            
Programming Language                   14       18.4
AI & Data Science                      11       14.5
Backend Development                    10       13.2
Cloud & DevOps        